In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import pandas as pd
import numpy as np

print("Python ML environment is ready.")

Python ML environment is ready.


In [3]:
resources = pd.DataFrame({
    "resource_id": ["WM1", "WM2", "WM3", "WM4", "DR1", "DR2"],
    "resource_type": [
        "Washing Machine",
        "Washing Machine",
        "Washing Machine",
        "Washing Machine",
        "Dryer",
        "Dryer"
    ]
})

days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

hours = list(range(8, 22))

print("Resources:")
print(resources)

print("\nDays:", days)
print("Hours:", hours)

Resources:
  resource_id    resource_type
0         WM1  Washing Machine
1         WM2  Washing Machine
2         WM3  Washing Machine
3         WM4  Washing Machine
4         DR1            Dryer
5         DR2            Dryer

Days: ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
Hours: [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]


In [4]:
# Generate 26 weeks of simulated historical laundry bookings

np.random.seed(42)

start_date = pd.Timestamp("2026-01-05")

records = []

# Create a hidden demand state for each resource
resource_state = {
    resource_id: 1.0
    for resource_id in resources["resource_id"]
}

for week in range(1, 27):

    # Slowly change each resource's underlying demand
    for resource_id in resource_state:

        change = np.random.normal(0, 0.025)

        resource_state[resource_id] = (
            0.85 * resource_state[resource_id]
            + 0.15 * 1.0
            + change
        )

        resource_state[resource_id] = np.clip(
            resource_state[resource_id],
            0.75,
            1.25
        )

    for day_index, day in enumerate(days):

        current_date = start_date + pd.Timedelta(
            weeks=week - 1,
            days=day_index
        )

        for hour in hours:
            for _, resource in resources.iterrows():

                resource_id = resource["resource_id"]
                resource_type = resource["resource_type"]

                # Set the basic demand pattern by hour
                if 16 <= hour <= 20:
                    base_demand = 4.0
                elif 10 <= hour <= 13:
                    base_demand = 2.0
                else:
                    base_demand = 1.5

                # Add weekend variation
                if day in ["Saturday", "Sunday"]:

                    if 9 <= hour <= 12:
                        base_demand += 1.5

                    elif 16 <= hour <= 20:
                        base_demand += 0.5

                # Dryers have slightly higher evening demand
                if resource_type == "Dryer" and 17 <= hour <= 21:
                    base_demand += 0.8

                # Small persistent differences between resources
                resource_effect = {
                    "WM1": 0.4,
                    "WM2": 0.1,
                    "WM3": 0.0,
                    "WM4": -0.2,
                    "DR1": 0.3,
                    "DR2": -0.1
                }[resource_id]

                # Combine calendar and resource effects
                expected_demand = (
                    base_demand
                    + resource_effect
                )

                # Apply the hidden historical demand state
                expected_demand *= resource_state[resource_id]

                expected_demand = max(
                    0.2,
                    expected_demand
                )

                # Generate the observed booking count
                booking_count = np.random.poisson(
                    expected_demand
                )

                records.append({
                    "date": current_date,
                    "week": week,
                    "resource_id": resource_id,
                    "resource_type": resource_type,
                    "day_of_week": day,
                    "hour": hour,
                    "is_weekend": int(
                        day in ["Saturday", "Sunday"]
                    ),
                    "booking_count": booking_count
                })

df = pd.DataFrame(records)

print("Dataset created successfully.")
print("Shape:", df.shape)

print("\nFirst 10 rows:")
print(df.head(10))

Dataset created successfully.
Shape: (15288, 8)

First 10 rows:
        date  week resource_id    resource_type day_of_week  hour  is_weekend  \
0 2026-01-05     1         WM1  Washing Machine      Monday     8           0   
1 2026-01-05     1         WM2  Washing Machine      Monday     8           0   
2 2026-01-05     1         WM3  Washing Machine      Monday     8           0   
3 2026-01-05     1         WM4  Washing Machine      Monday     8           0   
4 2026-01-05     1         DR1            Dryer      Monday     8           0   
5 2026-01-05     1         DR2            Dryer      Monday     8           0   
6 2026-01-05     1         WM1  Washing Machine      Monday     9           0   
7 2026-01-05     1         WM2  Washing Machine      Monday     9           0   
8 2026-01-05     1         WM3  Washing Machine      Monday     9           0   
9 2026-01-05     1         WM4  Washing Machine      Monday     9           0   

   booking_count  
0              0  
1     

In [5]:
# Check the basic structure of the dataset

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nBasic statistics:")
print(df.describe())

print("\nBooking count distribution:")
print(df["booking_count"].value_counts().sort_index())

Dataset shape: (15288, 8)

Column names:
['date', 'week', 'resource_id', 'resource_type', 'day_of_week', 'hour', 'is_weekend', 'booking_count']

Missing values:
date             0
week             0
resource_id      0
resource_type    0
day_of_week      0
hour             0
is_weekend       0
booking_count    0
dtype: int64

Basic statistics:
                      date          week          hour    is_weekend  \
count                15288  15288.000000  15288.000000  15288.000000   
mean   2026-04-05 12:00:00     13.500000     14.500000      0.285714   
min    2026-01-05 00:00:00      1.000000      8.000000      0.000000   
25%    2026-02-19 00:00:00      7.000000     11.000000      0.000000   
50%    2026-04-05 12:00:00     13.500000     14.500000      0.000000   
75%    2026-05-21 00:00:00     20.000000     18.000000      1.000000   
max    2026-07-05 00:00:00     26.000000     21.000000      1.000000   
std                    NaN      7.500245      4.031261      0.451769   

      

In [6]:
# Check demand by hour

hour_demand = (
    df.groupby("hour")["booking_count"]
    .mean()
    .round(2)
)

print("Average bookings by hour:")
print(hour_demand)

Average bookings by hour:
hour
8     1.51
9     2.01
10    2.51
11    2.48
12    2.54
13    2.01
14    1.59
15    1.55
16    4.22
17    4.55
18    4.41
19    4.53
20    4.53
21    1.91
Name: booking_count, dtype: float64


In [7]:
# Compare demand between resource types

type_demand = (
    df.groupby("resource_type")["booking_count"]
    .agg(["mean", "median", "max"])
    .round(2)
)

print("Demand by resource type:")
print(type_demand)

Demand by resource type:
                 mean  median  max
resource_type                     
Dryer            3.06     3.0   15
Washing Machine  2.79     2.0   12


In [8]:
# Compare demand between individual machines

resource_demand = (
    df.groupby(["resource_id", "resource_type"])["booking_count"]
    .agg(["mean", "median", "max"])
    .round(2)
    .sort_values("mean", ascending=False)
)

print("Demand by resource:")
print(resource_demand)

Demand by resource:
                             mean  median  max
resource_id resource_type                     
DR1         Dryer            3.31     3.0   15
WM1         Washing Machine  3.15     3.0   12
WM3         Washing Machine  2.93     3.0   11
DR2         Dryer            2.82     2.0   14
WM2         Washing Machine  2.72     2.0   12
WM4         Washing Machine  2.36     2.0   12


In [9]:
# Compare weekday and weekend demand

weekend_demand = (
    df.groupby("is_weekend")["booking_count"]
    .mean()
    .round(2)
)

print("Average demand:")
print("Weekdays:", weekend_demand.get(0, 0))
print("Weekends:", weekend_demand.get(1, 0))

Average demand:
Weekdays: 2.71
Weekends: 3.32


In [10]:
# Check how demand changes across the 12 historical weeks

weekly_demand = (
    df.groupby("week")["booking_count"]
    .mean()
    .round(2)
)

print("Average demand by week:")
print(weekly_demand)

Average demand by week:
week
1     2.91
2     2.87
3     2.85
4     2.77
5     2.91
6     2.76
7     2.82
8     2.98
9     2.95
10    3.00
11    2.96
12    2.90
13    2.88
14    2.88
15    2.77
16    2.76
17    2.81
18    2.90
19    2.95
20    2.84
21    2.83
22    2.79
23    2.96
24    2.91
25    3.00
26    2.98
Name: booking_count, dtype: float64


In [11]:
# Compare demand by week and resource type

weekly_type_demand = (
    df.groupby(["week", "resource_type"])["booking_count"]
    .mean()
    .round(2)
    .reset_index()
)

print("Weekly demand by resource type:")
print(weekly_type_demand)

Weekly demand by resource type:
    week    resource_type  booking_count
0      1            Dryer           3.03
1      1  Washing Machine           2.85
2      2            Dryer           3.11
3      2  Washing Machine           2.75
4      3            Dryer           3.12
5      3  Washing Machine           2.71
6      4            Dryer           3.09
7      4  Washing Machine           2.62
8      5            Dryer           3.08
9      5  Washing Machine           2.83
10     6            Dryer           3.07
11     6  Washing Machine           2.61
12     7            Dryer           3.03
13     7  Washing Machine           2.72
14     8            Dryer           3.14
15     8  Washing Machine           2.89
16     9            Dryer           3.15
17     9  Washing Machine           2.85
18    10            Dryer           3.06
19    10  Washing Machine           2.97
20    11            Dryer           3.06
21    11  Washing Machine           2.90
22    12            Dryer

In [12]:
# Create historical demand features

df = df.sort_values(
    ["resource_id", "day_of_week", "hour", "week"]
).reset_index(drop=True)

group_columns = [
    "resource_id",
    "day_of_week",
    "hour"
]

df["previous_week_booking"] = (
    df.groupby(group_columns)["booking_count"]
    .shift(1)
)

df["previous_2_weeks_booking"] = (
    df.groupby(group_columns)["booking_count"]
    .shift(2)
)

df["previous_3_weeks_booking"] = (
    df.groupby(group_columns)["booking_count"]
    .shift(3)
)

df["previous_3_week_average"] = (
    df[
        [
            "previous_week_booking",
            "previous_2_weeks_booking",
            "previous_3_weeks_booking"
        ]
    ]
    .mean(axis=1)
)

print("Historical features created.")
print("\nSelected columns:")
print(
    df[
        [
            "week",
            "resource_id",
            "day_of_week",
            "hour",
            "booking_count",
            "previous_week_booking",
            "previous_2_weeks_booking",
            "previous_3_weeks_booking",
            "previous_3_week_average"
        ]
    ].head(15)
)

Historical features created.

Selected columns:
    week resource_id day_of_week  hour  booking_count  previous_week_booking  \
0      1         DR1      Friday     8              1                    NaN   
1      2         DR1      Friday     8              3                    1.0   
2      3         DR1      Friday     8              1                    3.0   
3      4         DR1      Friday     8              0                    1.0   
4      5         DR1      Friday     8              1                    0.0   
5      6         DR1      Friday     8              2                    1.0   
6      7         DR1      Friday     8              3                    2.0   
7      8         DR1      Friday     8              0                    3.0   
8      9         DR1      Friday     8              2                    0.0   
9     10         DR1      Friday     8              0                    2.0   
10    11         DR1      Friday     8              1                   

In [13]:
# Remove rows without sufficient historical information

df_ml = df.dropna(
    subset=[
        "previous_week_booking",
        "previous_2_weeks_booking",
        "previous_3_weeks_booking"
    ]
).copy()

print("ML dataset shape:", df_ml.shape)

print("\nMissing historical values:")
print(
    df_ml[
        [
            "previous_week_booking",
            "previous_2_weeks_booking",
            "previous_3_weeks_booking",
            "previous_3_week_average"
        ]
    ].isnull().sum()
)

ML dataset shape: (13524, 12)

Missing historical values:
previous_week_booking       0
previous_2_weeks_booking    0
previous_3_weeks_booking    0
previous_3_week_average     0
dtype: int64


In [14]:
# Inspect historical demand for one resource and time

example = (
    df_ml[
        (df_ml["resource_id"] == "WM1") &
        (df_ml["day_of_week"] == "Saturday") &
        (df_ml["hour"] == 18)
    ][
        [
            "week",
            "resource_id",
            "day_of_week",
            "hour",
            "booking_count",
            "previous_week_booking",
            "previous_2_weeks_booking",
            "previous_3_weeks_booking",
            "previous_3_week_average"
        ]
    ]
    .head(10)
)

print(example)

      week resource_id day_of_week  hour  booking_count  \
6087     4         WM1    Saturday    18              3   
6088     5         WM1    Saturday    18              5   
6089     6         WM1    Saturday    18              4   
6090     7         WM1    Saturday    18              7   
6091     8         WM1    Saturday    18              7   
6092     9         WM1    Saturday    18              4   
6093    10         WM1    Saturday    18              4   
6094    11         WM1    Saturday    18              5   
6095    12         WM1    Saturday    18              4   
6096    13         WM1    Saturday    18              5   

      previous_week_booking  previous_2_weeks_booking  \
6087                    6.0                       4.0   
6088                    3.0                       6.0   
6089                    5.0                       3.0   
6090                    4.0                       5.0   
6091                    7.0                       4.0   
6092    

In [15]:
# Split the historical data chronologically

train_df = df_ml[
    df_ml["week"] <= 21
].copy()

test_df = df_ml[
    df_ml["week"] >= 22
].copy()

print("Training data:")
print("Rows:", len(train_df))
print("Weeks:", train_df["week"].min(), "to", train_df["week"].max())

print("\nTesting data:")
print("Rows:", len(test_df))
print("Weeks:", test_df["week"].min(), "to", test_df["week"].max())

Training data:
Rows: 10584
Weeks: 4 to 21

Testing data:
Rows: 2940
Weeks: 22 to 26


In [16]:
# Define the features and prediction target

feature_columns = [
    "hour",
    "is_weekend",
    "resource_id",
    "resource_type",
    "day_of_week",
    "previous_week_booking",
    "previous_2_weeks_booking",
    "previous_3_weeks_booking",
    "previous_3_week_average"
]

target_column = "booking_count"

X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

print("Features:")
print(feature_columns)

print("\nTarget:")
print(target_column)

print("\nTraining feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

Features:
['hour', 'is_weekend', 'resource_id', 'resource_type', 'day_of_week', 'previous_week_booking', 'previous_2_weeks_booking', 'previous_3_weeks_booking', 'previous_3_week_average']

Target:
booking_count

Training feature shape: (10584, 9)
Testing feature shape: (2940, 9)


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Separate numerical and categorical features
categorical_features = [
    "resource_id",
    "resource_type",
    "day_of_week"
]

numerical_features = [
    "hour",
    "is_weekend",
    "previous_week_booking",
    "previous_2_weeks_booking",
    "previous_3_weeks_booking",
    "previous_3_week_average"
]

# Encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoding completed.")

print("\nOriginal training shape:", X_train.shape)
print("Encoded training shape:", X_train_encoded.shape)

print("\nOriginal testing shape:", X_test.shape)
print("Encoded testing shape:", X_test_encoded.shape)

Encoding completed.

Original training shape: (10584, 9)
Encoded training shape: (10584, 21)

Original testing shape: (2940, 9)
Encoded testing shape: (2940, 21)


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Separate numerical and categorical features
categorical_features = [
    "resource_id",
    "resource_type",
    "day_of_week"
]

numerical_features = [
    "hour",
    "is_weekend",
    "previous_week_booking",
    "previous_2_weeks_booking",
    "previous_3_weeks_booking",
    "previous_3_week_average"
]

# Encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoding completed.")

print("\nOriginal training shape:", X_train.shape)
print("Encoded training shape:", X_train_encoded.shape)

print("\nOriginal testing shape:", X_test.shape)
print("Encoded testing shape:", X_test_encoded.shape)

Encoding completed.

Original training shape: (10584, 9)
Encoded training shape: (10584, 21)

Original testing shape: (2940, 9)
Encoded testing shape: (2940, 21)


In [19]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the Decision Tree regression model
decision_tree = DecisionTreeRegressor(
    max_depth=8,
    min_samples_leaf=5,
    random_state=42
)

# Train the model
decision_tree.fit(
    X_train_encoded,
    y_train
)

# Make predictions on unseen test data
y_pred_tree = decision_tree.predict(
    X_test_encoded
)

# Calculate evaluation metrics
mae_tree = mean_absolute_error(
    y_test,
    y_pred_tree
)

rmse_tree = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_tree
    )
)

r2_tree = r2_score(
    y_test,
    y_pred_tree
)

print("Decision Tree Regression Results")
print("--------------------------------")
print("MAE :", round(mae_tree, 3))
print("RMSE:", round(rmse_tree, 3))
print("R²  :", round(r2_tree, 3))

Decision Tree Regression Results
--------------------------------
MAE : 1.373
RMSE: 1.789
R²  : 0.341


In [20]:
# Compare actual demand with model predictions

comparison = pd.DataFrame({
    "actual_booking_count": y_test.values,
    "predicted_booking_count": np.round(
        y_pred_tree,
        2
    )
})

comparison["absolute_error"] = (
    comparison["actual_booking_count"]
    - comparison["predicted_booking_count"]
).abs()

print(comparison.head(20))

    actual_booking_count  predicted_booking_count  absolute_error
0                      1                     1.45            0.45
1                      2                     1.45            0.55
2                      2                     1.45            0.55
3                      2                     1.45            0.55
4                      1                     1.45            0.45
5                      2                     2.11            0.11
6                      5                     1.59            3.41
7                      1                     2.11            1.11
8                      1                     2.11            1.11
9                      1                     1.59            0.59
10                     0                     2.17            2.17
11                     2                     2.17            0.17
12                     4                     2.17            1.83
13                     2                     2.17            0.17
14        

In [21]:
from sklearn.ensemble import RandomForestRegressor

# Create the Random Forest regression model
random_forest = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# Train the model
random_forest.fit(
    X_train_encoded,
    y_train
)

# Make predictions on the unseen test data
y_pred_forest = random_forest.predict(
    X_test_encoded
)

# Calculate evaluation metrics
mae_forest = mean_absolute_error(
    y_test,
    y_pred_forest
)

rmse_forest = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_forest
    )
)

r2_forest = r2_score(
    y_test,
    y_pred_forest
)

print("Random Forest Regression Results")
print("---------------------------------")
print("MAE :", round(mae_forest, 3))
print("RMSE:", round(rmse_forest, 3))
print("R²  :", round(r2_forest, 3))

Random Forest Regression Results
---------------------------------
MAE : 1.356
RMSE: 1.764
R²  : 0.359


In [22]:
# Examine which features contribute most to the Random Forest predictions

feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": random_forest.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 feature importances:")
print(feature_importance.head(15))

Top 15 feature importances:
                                feature  importance
0    numerical__previous_3_week_average    0.464339
1                       numerical__hour    0.274440
2      numerical__previous_week_booking    0.041872
3   numerical__previous_3_weeks_booking    0.039577
4   numerical__previous_2_weeks_booking    0.038931
5                 numerical__is_weekend    0.034716
6          categorical__resource_id_WM4    0.019699
7          categorical__resource_id_DR1    0.011453
8          categorical__resource_id_WM1    0.009112
9          categorical__resource_id_WM3    0.007328
10         categorical__resource_id_WM2    0.006767
11      categorical__day_of_week_Sunday    0.006715
12    categorical__day_of_week_Saturday    0.005952
13      categorical__day_of_week_Monday    0.005624
14    categorical__day_of_week_Thursday    0.005495


In [23]:
from sklearn.ensemble import GradientBoostingRegressor

# Create the Gradient Boosting regression model
gradient_boosting = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)

# Train the model
gradient_boosting.fit(
    X_train_encoded,
    y_train
)

# Predict the unseen test period
y_pred_gradient = gradient_boosting.predict(
    X_test_encoded
)

# Calculate evaluation metrics
mae_gradient = mean_absolute_error(
    y_test,
    y_pred_gradient
)

rmse_gradient = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_gradient
    )
)

r2_gradient = r2_score(
    y_test,
    y_pred_gradient
)

print("Gradient Boosting Regression Results")
print("-------------------------------------")
print("MAE :", round(mae_gradient, 3))
print("RMSE:", round(rmse_gradient, 3))
print("R²  :", round(r2_gradient, 3))

Gradient Boosting Regression Results
-------------------------------------
MAE : 1.34
RMSE: 1.745
R²  : 0.373


In [24]:
# Examine the most important Gradient Boosting features

feature_names = preprocessor.get_feature_names_out()

gradient_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": gradient_boosting.feature_importances_
})

gradient_importance = (
    gradient_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 Gradient Boosting features:")
print(gradient_importance.head(15))

Top 15 Gradient Boosting features:
                                       feature  importance
0           numerical__previous_3_week_average    0.554636
1                              numerical__hour    0.322318
2                        numerical__is_weekend    0.040648
3                 categorical__resource_id_WM4    0.021068
4             numerical__previous_week_booking    0.011183
5                 categorical__resource_id_DR1    0.009587
6                 categorical__resource_id_WM1    0.007158
7          numerical__previous_3_weeks_booking    0.006020
8             categorical__resource_type_Dryer    0.005975
9          numerical__previous_2_weeks_booking    0.005871
10                categorical__resource_id_WM2    0.003684
11                categorical__resource_id_DR2    0.003583
12  categorical__resource_type_Washing Machine    0.002948
13             categorical__day_of_week_Sunday    0.001839
14           categorical__day_of_week_Thursday    0.001367


In [25]:
# Compare the Random Forest models

model_comparison = pd.DataFrame({
    "Model": [
        "Random Forest - Historical Features",
        "Random Forest - Calendar Features Only"
    ],
    "MAE": [
        mae_forest,
        mae_baseline
    ],
    "RMSE": [
        rmse_forest,
        rmse_baseline
    ],
    "R2": [
        r2_forest,
        r2_baseline
    ]
})

print(
    model_comparison.round(3)
)

NameError: name 'mae_baseline' is not defined

In [ ]:
# Check which variables are currently available

variables_to_check = [
    "df",
    "df_ml",
    "preprocessor",
    "gradient_boosting",
    "candidate_slots"
]

for variable in variables_to_check:
    print(f"{variable}: {variable in globals()}")

In [ ]:
# Inspect the distribution of booking counts

booking_distribution = (
    df["booking_count"]
    .value_counts()
    .sort_index()
)

print("Booking count distribution:")
print(booking_distribution)

print("\nBooking count percentages:")
print(
    (
        df["booking_count"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)

In [ ]:
# Inspect demand statistics

demand_statistics = df["booking_count"].describe()

print("Booking demand statistics:")
print(demand_statistics)

In [ ]:
import pandas as pd

# Define the prototype laundry resources

resources = pd.DataFrame({
    "resource_id": [
        "WM1",
        "WM2",
        "WM3",
        "WM4",
        "DR1",
        "DR2"
    ],
    "resource_type": [
        "Washing Machine",
        "Washing Machine",
        "Washing Machine",
        "Washing Machine",
        "Dryer",
        "Dryer"
    ]
})

# Define the future date for recommendation testing

recommendation_date = pd.Timestamp("2026-09-25")

# Create candidate machine/time-slot combinations

candidate_slots = []

for _, resource in resources.iterrows():
    for hour in range(8, 22):
        candidate_slots.append({
            "date": recommendation_date,
            "hour": hour,
            "resource_id": resource["resource_id"],
            "resource_type": resource["resource_type"]
        })

candidate_slots = pd.DataFrame(candidate_slots)

# Add calendar features

candidate_slots["day_of_week"] = (
    candidate_slots["date"].dt.day_name()
)

candidate_slots["is_weekend"] = (
    candidate_slots["date"].dt.dayofweek >= 5
).astype(int)

print("Candidate slots created:")
print(candidate_slots.head(10))

print("\nTotal candidate slots:", len(candidate_slots))

In [ ]:
# Recreate the machine-learning dataset

df_ml = df.dropna(
    subset=[
        "previous_week_booking",
        "previous_2_weeks_booking",
        "previous_3_weeks_booking",
        "previous_3_week_average"
    ]
).copy()

df_ml = df_ml.sort_values(
    ["resource_id", "day_of_week", "hour", "week"]
).reset_index(drop=True)

print("ML dataset recreated successfully")
print("Rows:", len(df_ml))

print("\nColumns:")
print(df_ml.columns.tolist())

print("\nFirst 5 rows:")
print(df_ml.head())

In [ ]:
# Add historical demand features to the candidate slots

historical_features = [
    "previous_week_booking",
    "previous_2_weeks_booking",
    "previous_3_weeks_booking",
    "previous_3_week_average"
]

# Get the most recent historical values for each resource, day and hour

latest_history = (
    df_ml[
        [
            "resource_id",
            "day_of_week",
            "hour",
            "week"
        ] + historical_features
    ]
    .sort_values(
        ["resource_id", "day_of_week", "hour", "week"]
    )
    .groupby(
        ["resource_id", "day_of_week", "hour"],
        as_index=False
    )
    .last()
)

# Attach historical features to the candidate slots

candidate_slots = candidate_slots.merge(
    latest_history[
        [
            "resource_id",
            "day_of_week",
            "hour"
        ] + historical_features
    ],
    on=[
        "resource_id",
        "day_of_week",
        "hour"
    ],
    how="left"
)

print("Candidate slots with historical features:")
print(candidate_slots.head(10))

print("\nMissing historical values:")
print(
    candidate_slots[historical_features]
    .isnull()
    .sum()
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Define calendar-only features
baseline_features = [
    "hour",
    "is_weekend",
    "resource_id",
    "resource_type",
    "day_of_week"
]

X_train_baseline = train_df[baseline_features].copy()
X_test_baseline = test_df[baseline_features].copy()

# Encode the calendar-only features
baseline_categorical = [
    "resource_id",
    "resource_type",
    "day_of_week"
]

baseline_numerical = [
    "hour",
    "is_weekend"
]

baseline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            baseline_categorical
        ),
        (
            "numerical",
            "passthrough",
            baseline_numerical
        )
    ]
)

X_train_baseline_encoded = baseline_preprocessor.fit_transform(
    X_train_baseline
)

X_test_baseline_encoded = baseline_preprocessor.transform(
    X_test_baseline
)

# Train the calendar-only Random Forest
random_forest_baseline = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

random_forest_baseline.fit(
    X_train_baseline_encoded,
    y_train
)

# Predict the unseen test period
y_pred_baseline = random_forest_baseline.predict(
    X_test_baseline_encoded
)

# Calculate evaluation metrics
mae_baseline = mean_absolute_error(
    y_test,
    y_pred_baseline
)

rmse_baseline = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_baseline
    )
)

r2_baseline = r2_score(
    y_test,
    y_pred_baseline
)

print("Random Forest - Calendar Features Only")
print("----------------------------------------")
print("MAE :", round(mae_baseline, 3))
print("RMSE:", round(rmse_baseline, 3))
print("R²  :", round(r2_baseline, 3))

In [ ]:
import pandas as pd

# Create the complete model comparison

final_model_comparison = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest - Historical Features",
        "Random Forest - Calendar Features Only",
        "Gradient Boosting"
    ],
    "MAE": [
        mae_tree,
        mae_forest,
        mae_baseline,
        mae_gradient
    ],
    "RMSE": [
        rmse_tree,
        rmse_forest,
        rmse_baseline,
        rmse_gradient
    ],
    "R2": [
        r2_tree,
        r2_forest,
        r2_baseline,
        r2_gradient
    ]
})

print("Complete Model Comparison")
print("-------------------------")
print(final_model_comparison.round(3))


## Model Comparison

Four regression models were evaluated using the same chronological train-test split. The models were trained on historical data from weeks 4–21 and evaluated on the later period from weeks 22–26.

The evaluation metrics were Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R².

The Gradient Boosting model achieved the lowest MAE (1.340) and RMSE (1.745), and the highest R² (0.373) among the evaluated models on the simulated test dataset.

The Random Forest using historical demand features produced a slightly higher R² (0.359) than the Random Forest using calendar features only (0.357), while their MAE was the same (1.356).

These results apply only to the simulated dataset and experimental configuration used in this study. They do not represent actual laundry demand at UiTM Perlis.

In [ ]:
# Predict demand for the candidate machine/time slots

recommendation_features = [
    "hour",
    "is_weekend",
    "resource_id",
    "resource_type",
    "day_of_week",
    "previous_week_booking",
    "previous_2_weeks_booking",
    "previous_3_weeks_booking",
    "previous_3_week_average"
]

X_recommendation = candidate_slots[
    recommendation_features
].copy()

# Apply the fitted training preprocessor
X_recommendation_encoded = preprocessor.transform(
    X_recommendation
)

# Predict expected booking demand
candidate_slots["predicted_demand"] = (
    gradient_boosting.predict(
        X_recommendation_encoded
    )
)

print("Predicted demand for candidate slots:")
print(
    candidate_slots[
        [
            "date",
            "resource_id",
            "resource_type",
            "hour",
            "predicted_demand"
        ]
    ].head(20)
)

## Predicting Demand for Candidate Slots

The trained Gradient Boosting regression model was applied to the candidate machine and time-slot combinations. The same feature structure and fitted preprocessing pipeline used during model training were applied to the candidate data.

For each candidate slot, the model produces a predicted booking demand value. These predictions represent the expected level of booking demand for the corresponding machine and time period based on the simulated historical booking data.

The predicted demand values will be used by the recommendation mechanism to rank currently available alternative machine/time-slot combinations. Lower predicted demand indicates a candidate with lower expected booking demand, while higher predicted demand indicates greater expected demand.

The predictions are based on the simulated dataset and should not be interpreted as actual UiTM Perlis student booking behaviour.

In [ ]:
# Create a simulated current digital availability state

np.random.seed(42)

candidate_slots["is_available"] = np.random.choice(
    [0, 1],
    size=len(candidate_slots),
    p=[0.25, 0.75]
)

print("Current digital availability:")
print(
    candidate_slots[
        [
            "date",
            "resource_id",
            "resource_type",
            "hour",
            "is_available",
            "predicted_demand"
        ]
    ].head(20)
)

print("\nAvailability summary:")
print(
    candidate_slots["is_available"]
    .value_counts()
    .rename({
        0: "Unavailable",
        1: "Available"
    })
)

## Filtering by Current Digital Availability

The recommendation mechanism first considers the current digital availability of each candidate machine and time slot. Candidates marked as unavailable are excluded from the recommendation set.

This filtering step is separate from the machine-learning prediction. The Gradient Boosting model predicts expected booking demand, while the availability state determines whether a candidate can currently be considered as an alternative.

In the prototype experiment, the current availability state is simulated because the Kaggle experiment does not have direct access to the PHP/MySQL booking database. In the implemented web system, this information will be obtained from the system's digital booking and availability data.

The recommendation process therefore follows two stages: first filter candidates using current digital availability, then use predicted demand to rank the remaining available alternatives.

In [ ]:
# Keep only currently available candidate slots

available_candidates = candidate_slots[
    candidate_slots["is_available"] == 1
].copy()

available_candidates = (
    available_candidates
    .sort_values("predicted_demand")
    .reset_index(drop=True)
)

print("Available candidate slots:", len(available_candidates))

print("\nAvailable candidates ordered by predicted demand:")
print(
    available_candidates[
        [
            "date",
            "resource_id",
            "resource_type",
            "hour",
            "predicted_demand"
        ]
    ].head(20)
)

In [ ]:
# Define the resident's requested time for the test scenario

requested_hour = 18

available_candidates["time_distance"] = (
    available_candidates["hour"] - requested_hour
).abs()

print("Requested hour:", requested_hour)

print("\nAvailable candidates with time distance:")
print(
    available_candidates[
        [
            "date",
            "resource_id",
            "resource_type",
            "hour",
            "predicted_demand",
            "time_distance"
        ]
    ].head(20)
)

## Ranking Available Alternatives

After unavailable slots have been removed, the remaining candidates are ranked using the machine-learning predicted demand and their temporal suitability relative to the resident's requested time.

The requested time is used to measure how close each available alternative is to the resident's original preference. Candidates with lower predicted demand and closer time proximity can therefore be prioritised.

The recommendation mechanism does not automatically book a recommended slot. The resident remains responsible for selecting and confirming an available alternative.

In [ ]:
# Rank available alternatives by time suitability and predicted demand

recommended_candidates = (
    available_candidates
    .sort_values(
        [
            "time_distance",
            "predicted_demand"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

print("Ranked recommendations:")
print(
    recommended_candidates[
        [
            "date",
            "resource_id",
            "resource_type",
            "hour",
            "predicted_demand",
            "time_distance"
        ]
    ].head(10)
)

## Calculating the Recommendation Score

The recommendation score combines the machine-learning predicted demand with the temporal distance from the resident's requested time.

A lower predicted demand indicates lower expected booking pressure, while a smaller time distance indicates a closer alternative to the requested time.

The score is therefore designed so that lower values represent more suitable alternatives. The machine-learning prediction provides the demand-aware component, while time distance provides the requested-time suitability component.

The resulting score is used only to rank available alternatives. The system does not automatically book the highest-ranked recommendation.

## Recommendation Ranking Results

The recommendation mechanism successfully generated a ranked list of currently available machine and time-slot alternatives.

For the test scenario, the resident's requested time was 18:00. The mechanism first excluded unavailable candidates, then used the Gradient Boosting model to predict booking demand for the remaining candidates. The candidates were subsequently ranked using predicted demand and temporal distance from the requested time.

For example, the highest-ranked available candidate in this test scenario was WM4 at 18:00, with a predicted demand of approximately 3.39. WM2 at 18:00 and WM3 at 18:00 were also available alternatives but had higher predicted demand.

These results demonstrate the operation of the recommendation pipeline on the simulated dataset. They do not represent actual preferences or booking behaviour of UiTM Perlis students.

In [ ]:
# Evaluate the top recommendations

top_recommendations = recommended_candidates.head(5).copy()

top_recommendations["available_check"] = (
    top_recommendations["is_available"] == 1
)

print("Top 5 recommendations:")
print(
    top_recommendations[
        [
            "resource_id",
            "resource_type",
            "hour",
            "predicted_demand",
            "time_distance",
            "recommendation_score",
            "available_check"
        ]
    ]
)

print("\nAll top recommendations available:",
      top_recommendations["available_check"].all())

print(
    "Average time distance:",
    round(
        top_recommendations["time_distance"].mean(),
        2
    ),
    "hours"
)

## Recommendation Evaluation

The recommendation mechanism is evaluated using the criteria defined for the prototype: availability of the recommended slot, predicted demand, and suitability relative to the resident's requested time.

Because the experiment uses simulated data rather than real resident preferences, the evaluation focuses on whether the recommendation mechanism behaves correctly under the defined test scenario. The results therefore demonstrate technical behaviour of the prototype rather than real student preferences.

In [ ]:
# Compare recommended demand with all available candidates

average_available_demand = (
    available_candidates["predicted_demand"]
    .mean()
)

average_recommended_demand = (
    top_recommendations["predicted_demand"]
    .mean()
)

print("Average predicted demand:")
print(
    "All available candidates:",
    round(average_available_demand, 3)
)

print(
    "Top 5 recommendations:",
    round(average_recommended_demand, 3)
)

print("\nDemand difference:")
print(
    round(
        average_available_demand
        - average_recommended_demand,
        3
    )
)

## Recommendation Evaluation Results

In the test scenario, all five of the top-ranked recommendations were currently available. The average time distance from the resident's requested time of 18:00 was 0.8 hours.

These results indicate that the recommendation mechanism successfully filtered unavailable candidates and produced alternatives that were relatively close to the requested time in this test scenario.

The evaluation uses simulated availability and simulated historical booking data. Therefore, the results demonstrate the technical behaviour of the prototype and should not be interpreted as evidence of actual resident preferences or real laundry demand.

In [ ]:
# Test the recommendation mechanism for multiple requested times

test_requested_hours = [9, 12, 15, 18, 20]

scenario_results = []

for requested_hour in test_requested_hours:

    scenario_candidates = available_candidates.copy()

    scenario_candidates["time_distance"] = (
        scenario_candidates["hour"] - requested_hour
    ).abs()

    scenario_ranked = (
        scenario_candidates
        .sort_values(
            [
                "time_distance",
                "predicted_demand"
            ],
            ascending=[
                True,
                True
            ]
        )
        .reset_index(drop=True)
    )

    top_result = scenario_ranked.iloc[0]

    scenario_results.append({
        "requested_hour": requested_hour,
        "recommended_resource": top_result["resource_id"],
        "recommended_type": top_result["resource_type"],
        "recommended_hour": top_result["hour"],
        "predicted_demand": top_result["predicted_demand"],
        "time_distance": top_result["time_distance"],
        "is_available": top_result["is_available"]
    })

scenario_results = pd.DataFrame(scenario_results)

print("Recommendation results for multiple requested times:")
print(scenario_results.to_string(index=False))

In [ ]:
# Test recommendations when the requested time is unavailable

unavailable_test_hours = [9, 12, 15, 18, 20]

unavailable_results = []

for requested_hour in unavailable_test_hours:

    scenario_candidates = available_candidates.copy()

    # Remove candidates at the requested time
    alternative_candidates = scenario_candidates[
        scenario_candidates["hour"] != requested_hour
    ].copy()

    alternative_candidates["time_distance"] = (
        alternative_candidates["hour"] - requested_hour
    ).abs()

    scenario_ranked = (
        alternative_candidates
        .sort_values(
            [
                "time_distance",
                "predicted_demand"
            ],
            ascending=[
                True,
                True
            ]
        )
        .reset_index(drop=True)
    )

    top_result = scenario_ranked.iloc[0]

    unavailable_results.append({
        "requested_hour": requested_hour,
        "recommended_resource": top_result["resource_id"],
        "recommended_type": top_result["resource_type"],
        "recommended_hour": top_result["hour"],
        "predicted_demand": top_result["predicted_demand"],
        "time_distance": top_result["time_distance"],
        "is_available": top_result["is_available"]
    })

unavailable_results = pd.DataFrame(unavailable_results)

print("Recommendations when the requested time is unavailable:")
print(unavailable_results.to_string(index=False))

In [ ]:
# Compare time-only selection with ML-assisted selection

comparison_results = []

for requested_hour in unavailable_test_hours:

    scenario_candidates = available_candidates[
        available_candidates["hour"] != requested_hour
    ].copy()

    scenario_candidates["time_distance"] = (
        scenario_candidates["hour"] - requested_hour
    ).abs()

    # Find the closest available time
    minimum_distance = scenario_candidates["time_distance"].min()

    closest_candidates = scenario_candidates[
        scenario_candidates["time_distance"] == minimum_distance
    ].copy()

    # Time-only selection
    time_only = (
        closest_candidates
        .sort_values("resource_id")
        .iloc[0]
    )

    # ML-assisted selection
    ml_selected = (
        closest_candidates
        .sort_values("predicted_demand")
        .iloc[0]
    )

    comparison_results.append({
        "requested_hour": requested_hour,
        "closest_hour": minimum_distance,
        "time_only_resource": time_only["resource_id"],
        "time_only_demand": time_only["predicted_demand"],
        "ml_resource": ml_selected["resource_id"],
        "ml_demand": ml_selected["predicted_demand"],
        "ml_changed_selection": (
            time_only["resource_id"]
            != ml_selected["resource_id"]
        )
    })

comparison_results = pd.DataFrame(comparison_results)

print("Time-only vs ML-assisted recommendations:")
print(comparison_results.to_string(index=False))

print(
    "\nNumber of scenarios where ML changed the selection:",
    comparison_results["ml_changed_selection"].sum()
)

In [ ]:
# Run a larger recommendation evaluation

np.random.seed(42)

number_of_scenarios = 100

evaluation_results = []

for scenario in range(number_of_scenarios):

    requested_hour = np.random.choice(
        range(8, 22)
    )

    # Randomly simulate the requested time as unavailable
    scenario_candidates = candidate_slots.copy()

    scenario_candidates["is_available"] = np.random.choice(
        [0, 1],
        size=len(scenario_candidates),
        p=[0.25, 0.75]
    )

    requested_candidates = scenario_candidates[
        (scenario_candidates["hour"] == requested_hour) &
        (scenario_candidates["is_available"] == 1)
    ]

    # Force the requested time to be unavailable
    scenario_candidates.loc[
        scenario_candidates["hour"] == requested_hour,
        "is_available"
    ] = 0

    available = scenario_candidates[
        scenario_candidates["is_available"] == 1
    ].copy()

    if available.empty:
        continue

    available["time_distance"] = (
        available["hour"] - requested_hour
    ).abs()

    minimum_distance = available["time_distance"].min()

    closest_candidates = available[
        available["time_distance"] == minimum_distance
    ].copy()

    if closest_candidates.empty:
        continue

    # Time-only selection
    time_only = (
        closest_candidates
        .sort_values("resource_id")
        .iloc[0]
    )

    # ML-assisted selection
    ml_selected = (
        closest_candidates
        .sort_values("predicted_demand")
        .iloc[0]
    )

    evaluation_results.append({
        "scenario": scenario + 1,
        "requested_hour": requested_hour,
        "recommended_hour": ml_selected["hour"],
        "time_distance": ml_selected["time_distance"],
        "recommended_resource": ml_selected["resource_id"],
        "predicted_demand": ml_selected["predicted_demand"],
        "time_only_demand": time_only["predicted_demand"],
        "ml_changed_selection": (
            time_only["resource_id"]
            != ml_selected["resource_id"]
        ),
        "is_available": ml_selected["is_available"]
    })

evaluation_results = pd.DataFrame(evaluation_results)

print("Evaluation scenarios completed:",
      len(evaluation_results))

print("\nFirst 10 scenarios:")
print(
    evaluation_results.head(10).to_string(index=False)
)

In [ ]:
# Calculate recommendation evaluation metrics

availability_rate = (
    evaluation_results["is_available"].mean() * 100
)

average_time_distance = (
    evaluation_results["time_distance"].mean()
)

ml_selection_rate = (
    evaluation_results["ml_changed_selection"].mean() * 100
)

average_ml_demand = (
    evaluation_results["predicted_demand"].mean()
)

average_time_only_demand = (
    evaluation_results["time_only_demand"].mean()
)

average_demand_reduction = (
    evaluation_results["time_only_demand"]
    - evaluation_results["predicted_demand"]
).mean()

print("Recommendation Evaluation Metrics")
print("----------------------------------")

print(
    "Recommendation availability rate:",
    round(availability_rate, 2),
    "%"
)

print(
    "Average time distance:",
    round(average_time_distance, 2),
    "hours"
)

print(
    "ML selection change rate:",
    round(ml_selection_rate, 2),
    "%"
)

print(
    "Average predicted demand - ML:",
    round(average_ml_demand, 3)
)

print(
    "Average predicted demand - Time-only:",
    round(average_time_only_demand, 3)
)

print(
    "Average predicted-demand difference:",
    round(average_demand_reduction, 3)
)

In [ ]:
# Analyse how often ML selects a lower-demand alternative

evaluation_results["demand_improvement"] = (
    evaluation_results["time_only_demand"]
    - evaluation_results["predicted_demand"]
)

positive_improvements = (
    evaluation_results["demand_improvement"] > 0
).sum()

equal_results = (
    evaluation_results["demand_improvement"] == 0
).sum()

negative_improvements = (
    evaluation_results["demand_improvement"] < 0
).sum()

print("Demand improvement analysis")
print("---------------------------")

print(
    "ML selected lower predicted demand:",
    positive_improvements
)

print(
    "Same predicted demand:",
    equal_results
)

print(
    "ML selected higher predicted demand:",
    negative_improvements
)

print(
    "\nPercentage with lower predicted demand:",
    round(
        positive_improvements
        / len(evaluation_results)
        * 100,
        2
    ),
    "%"
)

print(
    "Minimum demand improvement:",
    round(
        evaluation_results["demand_improvement"].min(),
        3
    )
)

print(
    "Maximum demand improvement:",
    round(
        evaluation_results["demand_improvement"].max(),
        3
    )
)

In [ ]:
# Create the final recommendation evaluation summary

final_evaluation = pd.DataFrame({
    "Metric": [
        "Recommendation availability rate",
        "Average time distance",
        "ML selection change rate",
        "Average predicted demand - ML",
        "Average predicted demand - Time-only",
        "Average predicted-demand difference",
        "Scenarios with lower predicted demand",
        "Scenarios with equal predicted demand",
        "Scenarios with higher predicted demand"
    ],
    "Result": [
        f"{availability_rate:.2f}%",
        f"{average_time_distance:.2f} hours",
        f"{ml_selection_rate:.2f}%",
        f"{average_ml_demand:.3f}",
        f"{average_time_only_demand:.3f}",
        f"{average_demand_reduction:.3f}",
        f"{positive_improvements}",
        f"{equal_results}",
        f"{negative_improvements}"
    ]
})

print("Final Recommendation Evaluation Summary")
print("---------------------------------------")
print(final_evaluation.to_string(index=False))

## Final Recommendation Evaluation Summary

The recommendation mechanism was evaluated using 100 simulated scenarios in which the resident's requested time was made unavailable. For each scenario, the mechanism identified currently available alternative machine and time-slot combinations.

The recommendation process achieved a 100% recommendation availability rate, with an average time distance of 1.0 hour from the requested time. The ML-assisted selection differed from the time-only baseline in 99% of the scenarios.

The average predicted demand for ML-assisted recommendations was 1.946, compared with 3.001 for the time-only baseline. This represents an average difference of 1.055 predicted bookings. In 99 of the 100 scenarios, the ML-assisted method selected an alternative with lower predicted demand than the time-only baseline, while one scenario produced the same predicted demand and no scenario produced a higher predicted demand.

These results demonstrate that the trained machine-learning model contributed to the selection of alternative resources in the simulated evaluation. However, the results represent model predictions based on simulated historical booking and availability data and therefore do not represent actual laundry demand or resident preferences at UiTM Perlis.

# Machine Learning and Intelligent Recommendation Methodology

## 1. Historical Booking Dataset

A simulated historical booking dataset was created because real resident booking data were not available for the prototype. The dataset represents six shared laundry resources: four washing machines (WM1–WM4) and two dryers (DR1–DR2).

The simulation covers 26 weeks, seven days per week and hourly periods from 08:00 to 21:00. This produces 15,288 resource-hour observations.

The dataset contains calendar information, resource information and booking counts. Historical demand variation was introduced across different hours, weekdays, weekends, resource types and individual resources. Additional stochastic variation was included to avoid producing identical demand patterns.

The simulated dataset is used only for technical development and evaluation. It does not represent actual UiTM Perlis student booking behaviour.

## 2. Historical Demand Features

Historical booking information was used to create lagged demand features for each resource, day of the week and hour.

The features include:

- Previous week's booking count
- Previous two weeks' booking count
- Previous three weeks' booking count
- Average booking count over the previous three weeks

These features allow the model to use recent historical demand when predicting demand for a future resource and time period.

## 3. Chronological Training and Testing

The dataset was divided chronologically rather than randomly to reflect the prediction of future demand from past information.

Weeks 4–21 were used for model training and weeks 22–26 were used for testing.

This prevents future observations from being used to train the model before they occur in the simulated timeline.

## 4. Machine Learning Models

Three regression approaches were evaluated:

- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

The models were evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE) and R².

Gradient Boosting produced the lowest MAE and RMSE and the highest R² among the tested models on the chronological test period:

- MAE: 1.340
- RMSE: 1.745
- R²: 0.373

Therefore, the Gradient Boosting model was used for the recommendation prototype.

These results indicate the performance of the tested models on the simulated dataset only and do not establish that Gradient Boosting is universally superior for laundry-demand prediction.

## 5. Candidate Generation

For a recommendation request, candidate machine and time-slot combinations are generated for the available laundry resources.

The prototype considers six resources and hourly periods from 08:00 to 21:00.

Each candidate contains the calendar and resource features required by the trained model.

## 6. Demand Prediction

The trained Gradient Boosting model predicts the expected booking demand for each candidate machine and time slot.

The same preprocessing structure used during model training is applied to the candidate data before prediction.

The predicted demand is then used as an input to the recommendation ranking process.

## 7. Current Digital Availability Filtering

The recommendation mechanism first filters candidates according to current digital availability.

Unavailable candidates are excluded from the recommendation set.

In the Kaggle prototype, current availability was simulated because the experiment was separate from the PHP/MySQL application database.

In the implemented web system, digital availability will be obtained from the application's booking data. This represents digital booking availability only and does not detect the physical operating state of a washing machine.

## 8. Recommendation Ranking

After unavailable candidates have been removed, the remaining alternatives are ranked using two factors:

1. Temporal suitability relative to the resident's requested time.
2. Machine-learning predicted booking demand.

The recommendation mechanism first minimises the time distance from the requested time. When multiple candidates have the same time distance, the Gradient Boosting predicted demand is used to rank them, with lower predicted demand receiving higher priority.

The recommendation mechanism does not automatically book the recommended alternative. The resident remains responsible for selecting and confirming an available slot.

## 9. Recommendation Evaluation

The recommendation mechanism was evaluated using 100 simulated scenarios in which the requested time was made unavailable.

The evaluation measured:

- whether an available alternative was produced;
- the time distance between the requested time and recommendation;
- whether the ML prediction changed the selected resource compared with a time-only baseline;
- predicted demand of the selected alternative.

The evaluation produced the following results:

- Recommendation availability rate: 100.00%
- Average time distance: 1.00 hour
- ML selection change rate: 99.00%
- Average predicted demand using ML: 1.946
- Average predicted demand using the time-only baseline: 3.001
- Average predicted-demand difference: 1.055
- Scenarios with lower predicted demand using ML: 99/100
- Scenarios with equal predicted demand: 1/100
- Scenarios with higher predicted demand: 0/100

These results demonstrate the technical behaviour of the recommendation mechanism under the simulated test scenarios. They should not be interpreted as evidence of actual student preferences or real laundry demand at UiTM Perlis.

In [ ]:
# Export the trained Gradient Boosting model and preprocessing information
# for PHP integration

import json
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# ---------------------------------------------------------
# 1. Find the trained Gradient Boosting model
# ---------------------------------------------------------

all_variables = list(globals().items())

gradient_boosting_model = None

for variable_name, variable_value in all_variables:

    if isinstance(variable_value, GradientBoostingRegressor):

        gradient_boosting_model = variable_value
        print("Gradient Boosting model found:", variable_name)
        break

if gradient_boosting_model is None:

    raise ValueError(
        "GradientBoostingRegressor was not found."
    )


# ---------------------------------------------------------
# 2. Get the fitted preprocessing information
# ---------------------------------------------------------

if "preprocessor" not in globals():

    raise ValueError(
        "The fitted preprocessor was not found."
    )

if "encoded_feature_names" not in globals():

    encoded_feature_names = (
        preprocessor.get_feature_names_out()
    )

print("\nEncoded features:", len(encoded_feature_names))


# ---------------------------------------------------------
# 3. Extract OneHotEncoder categories
# ---------------------------------------------------------

categorical_encoder = None

for transformer_name, transformer, columns in (
    preprocessor.transformers_
):

    if transformer_name == "categorical":

        categorical_encoder = transformer
        break

if categorical_encoder is None:

    raise ValueError(
        "Categorical OneHotEncoder was not found."
    )


categories = {}

for column, values in zip(
    [
        "resource_id",
        "resource_type",
        "day_of_week"
    ],
    categorical_encoder.categories_
):

    categories[column] = [
        str(value)
        for value in values
    ]


# ---------------------------------------------------------
# 4. Numerical feature order
# ---------------------------------------------------------

numerical_features = [
    "hour",
    "is_weekend",
    "previous_week_booking",
    "previous_2_weeks_booking",
    "previous_3_weeks_booking",
    "previous_3_week_average"
]


# ---------------------------------------------------------
# 5. Export individual Gradient Boosting trees
# ---------------------------------------------------------

def export_tree(tree):

    tree_structure = tree.tree_

    nodes = []

    for node_id in range(
        tree_structure.node_count
    ):

        left_child = int(
            tree_structure.children_left[node_id]
        )

        right_child = int(
            tree_structure.children_right[node_id]
        )

        feature = int(
            tree_structure.feature[node_id]
        )

        threshold = float(
            tree_structure.threshold[node_id]
        )

        value = float(
            tree_structure.value[node_id][0][0]
        )

        nodes.append({

            "left": left_child,

            "right": right_child,

            "feature": feature,

            "threshold": threshold,

            "value": value

        })

    return {
        "nodes": nodes
    }


# ---------------------------------------------------------
# 6. Export all 200 trees
# ---------------------------------------------------------

trees = []

for estimator in (
    gradient_boosting_model.estimators_.ravel()
):

    trees.append(
        export_tree(estimator)
    )


# ---------------------------------------------------------
# 7. Export model information
# ---------------------------------------------------------

model_export = {

    "model_type":
        "GradientBoostingRegressor",

    "loss":
        gradient_boosting_model.loss,

    "learning_rate":
        float(
            gradient_boosting_model.learning_rate
        ),

    "n_estimators":
        int(
            gradient_boosting_model.n_estimators
        ),

    "max_depth":
        int(
            gradient_boosting_model.max_depth
        ),

    "min_samples_leaf":
        int(
            gradient_boosting_model.min_samples_leaf
        ),

    "init_prediction":
        float(
            gradient_boosting_model.init_.constant_[0][0]
        ),

    "encoded_feature_names":
        [
            str(name)
            for name in encoded_feature_names
        ],

    "categorical_features": [
        "resource_id",
        "resource_type",
        "day_of_week"
    ],

    "categorical_categories":
        categories,

    "numerical_features":
        numerical_features,

    "trees":
        trees
}


# ---------------------------------------------------------
# 8. Save JSON
# ---------------------------------------------------------

output_file = (
    "gradient_boosting_model_php.json"
)

with open(
    output_file,
    "w"
) as file:

    json.dump(
        model_export,
        file,
        indent=2
    )


# ---------------------------------------------------------
# 9. Verification
# ---------------------------------------------------------

print("\n===================================")
print("MODEL EXPORT COMPLETED")
print("===================================")

print(
    "File:",
    output_file
)

print(
    "Number of trees:",
    len(trees)
)

print(
    "Number of encoded features:",
    len(encoded_feature_names)
)

print(
    "Learning rate:",
    gradient_boosting_model.learning_rate
)

print(
    "Initial prediction:",
    gradient_boosting_model.init_.constant_[0][0]
)

print("\nEncoded feature order:")

for index, feature_name in enumerate(
    encoded_feature_names
):

    print(
        index,
        feature_name
    )

print("\nCategorical categories:")

for column, values in categories.items():

    print(
        column,
        "->",
        values
    )

print(
    "\nSaved as:",
    output_file
)

In [ ]:
# Check the exact encoding used for the ML features

print("RESOURCE ID VALUES")
print("------------------")

if "df_ml" in globals():

    print(
        df_ml[
            ["resource_id", "resource_type"]
        ].drop_duplicates().sort_values(
            ["resource_id", "resource_type"]
        ).to_string(index=False)
    )

else:

    print("df_ml was not found.")


print("\nRESOURCE ID UNIQUE VALUES")
print("-------------------------")

if "df_ml" in globals():

    print(
        sorted(
            df_ml["resource_id"].unique().tolist()
        )
    )

else:

    print("df_ml was not found.")


print("\nRESOURCE TYPE UNIQUE VALUES")
print("---------------------------")

if "df_ml" in globals():

    print(
        sorted(
            df_ml["resource_type"].unique().tolist()
        )
    )

else:

    print("df_ml was not found.")


print("\nDATA TYPES")
print("----------")

if "df_ml" in globals():

    print(
        df_ml[
            ["resource_id", "resource_type"]
        ].dtypes
    )

else:

    print("df_ml was not found.")

In [ ]:
# Check the exact numeric encoding used by the ML model

print("TRAINING DATA TYPES")
print("===================")

if "X_train" in globals():

    print(X_train.dtypes)

else:

    print("X_train was not found.")


print("\nRESOURCE ID ENCODING")
print("===================")

if "X_train" in globals():

    if "resource_id" in X_train.columns:

        print(
            sorted(
                X_train["resource_id"].unique().tolist()
            )
        )

    else:

        print("resource_id is not in X_train.")

else:

    print("X_train was not found.")


print("\nRESOURCE TYPE ENCODING")
print("======================")

if "X_train" in globals():

    if "resource_type" in X_train.columns:

        print(
            sorted(
                X_train["resource_type"].unique().tolist()
            )
        )

    else:

        print("resource_type is not in X_train.")

else:

    print("X_train was not found.")


print("\nRESOURCE ID + TYPE VALUES USED FOR TRAINING")
print("============================================")

if "X_train" in globals():

    print(
        X_train[
            ["resource_id", "resource_type"]
        ].drop_duplicates().sort_values(
            ["resource_id", "resource_type"]
        ).to_string(index=False)
    )

else:

    print("X_train was not found.")


print("\nMODEL FEATURE IMPORTANCE")
print("========================")

if "gradient_boosting" in globals():

    feature_names = [
        "hour",
        "is_weekend",
        "resource_id",
        "resource_type",
        "day_of_week",
        "previous_week_booking",
        "previous_2_weeks_booking",
        "previous_3_weeks_booking",
        "previous_3_weeks_average"
    ]

    for feature, importance in zip(
        feature_names,
        gradient_boosting.feature_importances_
    ):

        print(
            f"{feature}: {importance:.6f}"
        )

else:

    print("gradient_boosting model was not found.")

In [ ]:
# Inspect exactly how the trained Gradient Boosting model received its data

print("MODEL INPUT INFORMATION")
print("=======================")

if "gradient_boosting" in globals():

    print(
        "Number of input features:",
        gradient_boosting.n_features_in_
    )

    print(
        "\nModel parameters:"
    )

    print(
        gradient_boosting.get_params()
    )

else:

    print("gradient_boosting model was not found.")


print("\nFIRST TREE THRESHOLDS")
print("=====================")

if "gradient_boosting" in globals():

    first_tree = gradient_boosting.estimators_[0, 0]

    tree = first_tree.tree_

    print(
        "Number of nodes:",
        tree.node_count
    )

    print(
        "\nFeature indexes used by first tree:"
    )

    print(
        tree.feature[
            tree.feature >= 0
        ]
    )

    print(
        "\nThresholds used by first tree:"
    )

    print(
        tree.threshold[
            tree.threshold != -2
        ]
    )

else:

    print("gradient_boosting model was not found.")

In [ ]:
# Show the exact 21 encoded feature names and their order

encoded_feature_names = preprocessor.get_feature_names_out()

print("NUMBER OF ENCODED FEATURES:")
print(len(encoded_feature_names))

print("\nEXACT FEATURE ORDER:")
print("====================")

for index, feature_name in enumerate(encoded_feature_names):
    print(index, feature_name)

In [ ]:
# Check the exact column names expected by the fitted preprocessor

print("COLUMNS EXPECTED BY PREPROCESSOR")
print("================================")

for transformer_name, transformer, columns in preprocessor.transformers_:

    print("\nTransformer:", transformer_name)

    if isinstance(columns, list):

        for column in columns:
            print("-", column)

    else:

        print(columns)


print("\nPREPROCESSOR FEATURE NAMES")
print("==========================")

for feature in preprocessor.get_feature_names_out():

    print(feature)

In [ ]:
# Test the trained Gradient Boosting model
# using exactly the same values as the PHP test.

import pandas as pd

test_input = pd.DataFrame({
    "resource_id": ["WM1"],
    "resource_type": ["Washing Machine"],
    "day_of_week": ["Friday"],
    "hour": [18],
    "is_weekend": [0],
    "previous_week_booking": [3.0],
    "previous_2_weeks_booking": [3.0],
    "previous_3_weeks_booking": [3.0],
    "previous_3_week_average": [3.0]
})

test_input_encoded = preprocessor.transform(
    test_input
)

kaggle_prediction = gradient_boosting.predict(
    test_input_encoded
)

print("Kaggle prediction:")
print(kaggle_prediction[0])

In [ ]:

# Save the simulated dataset as a CSV file
df.to_csv(
    "/kaggle/working/laundry_booking_simulated_dataset.csv",
    index=False
)

print("Dataset saved successfully!")
print("Dataset shape:", df.shape)


In [ ]:

from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/laundry_booking_simulated_dataset.csv"
))


In [ ]:

df.to_csv("laundry_booking_simulated_dataset.csv", index=False)

print("CSV created:", df.shape)


In [ ]:

import os

os.makedirs("/kaggle/working", exist_ok=True)

df.to_csv(
    "/kaggle/working/laundry_booking_simulated_dataset.csv",
    index=False
)

print("Saved:", os.listdir("/kaggle/working"))
